##a. LIBRARY INSTALL AND IMPORT

In [ ]:
!pip install pandas numpy scikit-learn torch transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
# This cell will authenticate you and mount your Drive in the Colab.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Import all necessary libraries
import re
import nltk
import os
import json
import pandas as pd
import torch
import numpy as np
from sklearn.utils import resample
from transformers import BertTokenizer, BertModel
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
# check some numpy version
print(f"NumPy version: {np.__version__}")

NumPy version: 2.0.2


#b.DATA

###b1.DATA LOADING

In [ ]:
import json
import pandas as pd

# Function to read a JSON file
def load_json_file(file_path):
    """Load a JSON file"""
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

# Function to process FakeNewsNet data
def process_FakeNewsNet_data(file_paths):
    """Read multiple JSON files and organize them into a DataFrame"""
    data_list = []  # Store all data

    for category, file_path in file_paths.items():
        data = load_json_file(file_path)  # Load JSON data

        for news_id, news_content in data.items():
            if 'text' in news_content:  # Ensure the text field exists
                # Determine the news source
                if "gossipcop" in category:
                    source = "gossipcop"
                elif "politifact" in category:
                    source = "politifact"
                else:
                    source = "unknown"

                # Determine the news type
                if "R" in category:
                    if "HR" in category:
                        label = "human_real" # Human-written real news
                    else:
                        label = "gpt_real"  # AI-generated real news
                elif "F" in category:
                    if "MF" in category:
                        label = "gpt_fake"  # AI-generated fake news
                    else:
                        label = "human_fake"  # Human-written fake news
                else:
                    label = "unknown"

                data_list.append({
                    "id": news_id,
                    "text": news_content["text"],
                    "title": news_content.get("title", ""),  # Some data may not have a title
                    "source": source,  # News source
                    "label": label,  # Real/fake news category
                })

    # Convert to a Pandas DataFrame
    df = pd.DataFrame(data_list)
    return df

# Define paths for the dataset
fake_news_paths = {
    "gossipcop_HF": "/content/drive/MyDrive/DS266/HF.json",
    "gossipcop_HR": "/content/drive/MyDrive/DS266/HR.json",
    "gossipcop_MF": "/content/drive/MyDrive/DS266/MF.json",
    "gossipcop_MR": "/content/drive/MyDrive/DS266/MR.json",
}

# Load and process the data
FakeNewsNet_df = process_FakeNewsNet_data(fake_news_paths)

# Display the first 5 rows
FakeNewsNet_df.head()


,id,text,title,source,label
0,0,✕ Close Meghan Markle and Prince Harry have an...,As it happened: Prince Harry and Meghan Markle...,gossipcop,human_fake
1,1,Kim Kardashian and Kanye West are pulling out ...,Kim & Kanye Install At-Home Panic Room After P...,gossipcop,human_fake
2,2,Prince Harry and Meghan currently live at Kens...,£1.4million spent renovating Prince Harry and ...,gossipcop,human_fake
3,3,They can't get enough of the Biebs on this sho...,Photos from Dancing With the Stars: Special Gu...,gossipcop,human_fake
4,4,Ben Affleck is keeping life with his three kid...,Jennifer Garner ‘Doesn’t Want’ Her Kids Around...,gossipcop,human_fake


# c.Model Optmization
Direction
1. gpt fake & gpt real have shorter text, balance the length
  1.  reduce current length
  2.  data augmentation
2. humanize gpt text by adding typo for gpt real & fake
3. two stage model



In [ ]:

!pip install -q transformers
!pip install -q torchinfo
!pip install -q datasets
!pip install -q evaluate

In [ ]:

import numpy as np

import transformers
import evaluate

from datasets import load_dataset
from torchinfo import summary

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import nltk
nltk.download('stopwords')
nltk.download('punkt')

from datasets import Dataset
import pandas as pd

from sklearn.metrics import confusion_matrix, classification_report
import numpy as np


from transformers import Trainer, TrainingArguments
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


#### C1: gpt fake & gpt real have shorter text, balance the length

In [ ]:
# combine text and title for training
FakeNewsNet_df['combined_text'] = FakeNewsNet_df['text'] + '.' + FakeNewsNet_df['title']

FakeNewsNet_df["word_count"] = FakeNewsNet_df["combined_text"].apply(lambda x: len(x.split()))


MR =  FakeNewsNet_df[(FakeNewsNet_df['label'] == "gpt_real") & (FakeNewsNet_df['word_count'] < 500) & (FakeNewsNet_df['word_count'] > 80)]
MF =  FakeNewsNet_df[(FakeNewsNet_df['label'] == "gpt_fake") & (FakeNewsNet_df['word_count'] < 500) & (FakeNewsNet_df['word_count'] > 80)]
HR =  FakeNewsNet_df[(FakeNewsNet_df['label'] == "human_real") & (FakeNewsNet_df['word_count'] < 500) & (FakeNewsNet_df['word_count'] > 80)]
HF =  FakeNewsNet_df[(FakeNewsNet_df['label'] == "human_fake") & (FakeNewsNet_df['word_count'] < 500) & (FakeNewsNet_df['word_count'] > 80)]


new_FNN_df = pd.concat([MR,MF,HR,HF])

print(new_FNN_df['label'].value_counts())

##lowest is 2756

label
human_real    4853
gpt_fake      4080
gpt_real      3527
human_fake    2756
Name: count, dtype: int64


In [ ]:
MR = resample(MR,replace=True,n_samples=2756,random_state=42)
MF = resample(MF,replace=True,n_samples=2756,random_state=42)
HR = resample(HR,replace=True,n_samples=2756,random_state=42)


Bal_FNN_df = pd.concat([MR,MF,HR,HF])

print(Bal_FNN_df['label'].value_counts())



label
gpt_real      2756
gpt_fake      2756
human_real    2756
human_fake    2756
Name: count, dtype: int64


In [ ]:
# Convert labels to numerical values (0=real, 1=fake)
Bal_FNN_df['label_new'] = Bal_FNN_df['label'].map({'human_fake': 0, 'human_real': 1.0,'gpt_fake': 2,'gpt_real':3})

# calcalate word count distributon of news text
Bal_FNN_df["word_count"] = Bal_FNN_df["combined_text"].apply(lambda x: len(x.split()))
Bal_FNN_df.groupby("label")["word_count"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
gpt_fake,2756.0,235.175617,52.481711,85.0,201.0,233.0,267.0,496.0
gpt_real,2756.0,327.410377,83.272611,84.0,270.0,327.0,386.0,499.0
human_fake,2756.0,295.289550,119.029550,81.0,192.0,316.0,393.0,499.0
human_real,2756.0,301.905298,107.708082,81.0,220.0,308.0,387.0,499.0


In [ ]:
import string

def simple_tokenizer(text):
    return re.findall(r'\b\w+\b', text.lower())

text = "Hello! This is a test."
print(simple_tokenizer(text))

# Remove unwanted characters, links, and HTML tags
def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove special characters and numbers, keeping only letters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Convert to lowercase
    text = text.lower()
    words = simple_tokenizer(text)
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)



['hello', 'this', 'is', 'a', 'test']


In [ ]:
# Apply cleaning to datasets
Bal_FNN_df['cleaned_combined_text'] = Bal_FNN_df['combined_text'].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data while keeping the DataFrame structure ( for hugging face dataset use)

train_df, tmp_df = train_test_split(
    Bal_FNN_df[['cleaned_combined_text', 'label_new']],
    test_size=0.4,
    random_state=42,
    stratify=Bal_FNN_df['label_new']
)

val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.5,
    random_state=42,
    stratify=tmp_df['label_new']
)
# Reset the index for both DataFrames
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Val samples: {len(test_df)}")
print("\nTrain DataFrame:")
print(train_df.head())
print("\nTest DataFrame:")
print(test_df.head())
print("\nVal DataFrame:")
print(val_df.head())

Training samples: 6614
Test samples: 2205
Val samples: 2205

Train DataFrame:
                               cleaned_combined_text  label_new
0  chrissy teigen john legend fair share ups down...        3.0
1  kourtney kardashians boyfriend younes bendjima...        1.0
2  former nfl star terrell owens announced good m...        1.0
3  katharine mcphee david foster engaged hollywoo...        1.0
4  halloween might still weeks away counting anyt...        1.0

Test DataFrame:
                               cleaned_combined_text  label_new
0  urassic world star chris pratt spotted coffee ...        2.0
1  brad pitt angelina jolie ever going get differ...        0.0
2  comes certain television shows sometimes choos...        1.0
3  mila kunis renowned hollywood actress finally ...        2.0
4  woman hanging drake split jennifer lopez claim...        0.0

Val DataFrame:
                               cleaned_combined_text  label_new
0  kardashians celebrated americas independence d...     

In [ ]:
pip cache purge

Files removed: 198


In [ ]:
!pip install checklist

In [ ]:
from checklist.perturb import Perturb

def safe_add_typos(text, min_length=3, max_typos=2):
    if pd.isna(text) or len(text) < min_length:
        return text

    try:
        n_typos = min(max_typos, len(text)//2)  # Max 1 typo per 2 characters
        return Perturb.add_typos(text)
    except ValueError:
        return text  # Fallback for edge cases

def conditional_add_typos(row):
        return safe_add_typos(row['cleaned_combined_text'])




train_df['text_add_typos'] = train_df.apply(conditional_add_typos, axis=1)

# test_df['text_add_typos'] = test_df.apply(conditional_add_typos, axis=1)

# val_df['text_add_typos'] = val_df.apply(conditional_add_typos, axis=1)


# train_df['text_add_typos'] = train_df['cleaned_combined_text'].apply(safe_add_typos)

# # Verify
# empty_count = train_df['cleaned_combined_text'].apply(lambda x: len(x) < 3).sum()
# print(f"Fixed {empty_count} short/empty texts")

In [ ]:
train_df.drop(columns=['cleaned_combined_text'], inplace=True)
# test_df.drop(columns=['cleaned_combined_text'], inplace=True)
# val_df.drop(columns=['cleaned_combined_text'], inplace=True)

In [ ]:
# Convert to Hugging Face Dataset for training
formatted_df = train_df.rename(columns={
    'text_add_typos': 'text',
    'label_new': 'label'
})

fake_news_dataset_train = Dataset.from_pandas(formatted_df)

# Convert to Hugging Face Dataset for testimg
formatted_df = test_df.rename(columns={
    'cleaned_combined_text': 'text',
    'label_new': 'label'
})

fake_news_dataset_test = Dataset.from_pandas(formatted_df)

# Convert to Hugging Face Dataset for validating
formatted_df = val_df.rename(columns={
    'cleaned_combined_text': 'text',
    'label_new': 'label'
})

fake_news_dataset_val = Dataset.from_pandas(formatted_df)

print("Test: \n",fake_news_dataset_test)
print("Train: \n",fake_news_dataset_train)
print("Val: \n",fake_news_dataset_val)

Test: 
 Dataset({
    features: ['text', 'label'],
    num_rows: 2205
})
Train: 
 Dataset({
    features: ['label', 'text'],
    num_rows: 6614
})
Val: 
 Dataset({
    features: ['text', 'label'],
    num_rows: 2205
})


In [ ]:
MAX_SEQUENCE_LENGTH = 496

In [ ]:
# Encode data

def preprocess_data(data, tokenizer):
    review_text = data['text']

    encoded = tokenizer.batch_encode_plus(
            review_text,
            max_length=MAX_SEQUENCE_LENGTH,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors="pt"
        )

    return encoded

In [ ]:
#Creating  compute metrics function

metric = evaluate.load('accuracy')

# FakeNewsNet_df['label_new'] = FakeNewsNet_df['label'].map({'human_fake': 0, 'human_real': 1.0,'gpt_fake': 2,'gpt_real': 3})
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)
  precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
  acc = accuracy_score(labels, predictions)

  class_report = classification_report(
        labels, predictions,
        target_names=['human_fake', 'human_real', 'gpt_fake','gpt_real'],
        digits=4
    )
  print("\nClassification Report:")
  print(class_report)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }


In [ ]:
## Fine tuning the "bert-base-cased" model

def fine_tune_classification_model(classification_model,
                                   tokenizer,
                                   train_data,
                                   val_data,
                                   batch_size = 16,
                                   num_epochs = 3):

    preprocessed_train_data = train_data.map(preprocess_data, batched=True, fn_kwargs={'tokenizer': tokenizer})
    preprocessed_val_data = val_data.map(preprocess_data, batched=True, fn_kwargs={'tokenizer': tokenizer})

    preprocessed_train_data = preprocessed_train_data.map(
        lambda x: {'labels': torch.tensor(x['label'], dtype=torch.long)}  # Ensure labels are int64
    )
    preprocessed_val_data  = preprocessed_val_data .map(
        lambda x: {'labels': torch.tensor(x['label'], dtype=torch.long)}
    )



    training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/DS266/model_output",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_dir="/content/drive/MyDrive/DS266/logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to='none'
)
    trainer = Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=preprocessed_train_data,
        eval_dataset=preprocessed_val_data,
        compute_metrics=compute_metrics
    )


    trainer.train()



### bert-base model

In [ ]:
model_checkpoint_name = "bert-base-cased"
bert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
# bert_classification_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint_name)

bert_classification_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint_name,
    num_labels=4  # number of classes = 4
)

# fine_tune_classification_model(bert_classification_model, bert_tokenizer, fake_news_dataset_train, fake_news_dataset_val)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
fine_tune_classification_model(bert_classification_model, bert_tokenizer, fake_news_dataset_train, fake_news_dataset_val)

Map:   0%|          | 0/6614 [00:00<?, ? examples/s]

Map:   0%|          | 0/2205 [00:00<?, ? examples/s]

Map:   0%|          | 0/6614 [00:00<?, ? examples/s]

Map:   0%|          | 0/2205 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.441366,0.835374,0.834411,0.839385,0.835374
2,0.667400,0.516364,0.841723,0.837726,0.842952,0.841723
3,0.313100,0.449682,0.864853,0.863208,0.864570,0.864853



Classification Report:
              precision    recall  f1-score   support

  human_fake     0.8333    0.6806    0.7493       551
  human_real     0.7159    0.8276    0.7677       551
    gpt_fake     0.9221    0.9438    0.9329       552
    gpt_real     0.8861    0.8893    0.8877       551

    accuracy                         0.8354      2205
   macro avg     0.8393    0.8353    0.8344      2205
weighted avg     0.8394    0.8354    0.8344      2205


Classification Report:
              precision    recall  f1-score   support

  human_fake     0.8665    0.6715    0.7566       551
  human_real     0.7768    0.7895    0.7831       551
    gpt_fake     0.8423    0.9873    0.9091       552
    gpt_real     0.8862    0.9183    0.9020       551

    accuracy                         0.8417      2205
   macro avg     0.8430    0.8417    0.8377      2205
weighted avg     0.8430    0.8417    0.8377      2205


Classification Report:
              precision    recall  f1-score   support

  h

### roberta-base model

In [ ]:
model_checkpoint_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
roberta_classification_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint_name,
    num_labels=4  # number of classes = 4
)



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
fine_tune_classification_model(roberta_classification_model, roberta_tokenizer, fake_news_dataset_train, fake_news_dataset_val)

Map:   0%|          | 0/6614 [00:00<?, ? examples/s]

Map:   0%|          | 0/2205 [00:00<?, ? examples/s]

Map:   0%|          | 0/6614 [00:00<?, ? examples/s]

Map:   0%|          | 0/2205 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.580030,0.794558,0.791571,0.809912,0.794558
2,0.616500,0.466318,0.847166,0.845758,0.849784,0.847166
3,0.309000,0.446150,0.867120,0.866443,0.866679,0.867120



Classification Report:
              precision    recall  f1-score   support

  human_fake     0.7436    0.7895    0.7658       551
  human_real     0.7537    0.7387    0.7461       551
    gpt_fake     0.7770    0.9909    0.8710       552
    gpt_real     0.9654    0.6588    0.7832       551

    accuracy                         0.7946      2205
   macro avg     0.8099    0.7945    0.7915      2205
weighted avg     0.8099    0.7946    0.7916      2205


Classification Report:
              precision    recall  f1-score   support

  human_fake     0.8205    0.7550    0.7864       551
  human_real     0.7930    0.7858    0.7894       551
    gpt_fake     0.8341    0.9928    0.9065       552
    gpt_real     0.9515    0.8548    0.9006       551

    accuracy                         0.8472      2205
   macro avg     0.8498    0.8471    0.8457      2205
weighted avg     0.8498    0.8472    0.8458      2205


Classification Report:
              precision    recall  f1-score   support

  h

### distilbert model

In [ ]:
model_checkpoint_name = "distilbert-base-uncased"
distilbert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
distilbert_classification_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint_name,
    num_labels=4  # number of classes = 4
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
fine_tune_classification_model(distilbert_classification_model, distilbert_tokenizer, fake_news_dataset_train, fake_news_dataset_val)

Map:   0%|          | 0/6614 [00:00<?, ? examples/s]

Map:   0%|          | 0/2205 [00:00<?, ? examples/s]

Map:   0%|          | 0/6614 [00:00<?, ? examples/s]

Map:   0%|          | 0/2205 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.598814,0.775057,0.770094,0.792618,0.775057
2,0.605200,0.385334,0.865306,0.864535,0.866859,0.865306
3,0.261800,0.430280,0.865760,0.864796,0.864766,0.865760



Classification Report:
              precision    recall  f1-score   support

  human_fake     0.6859    0.8403    0.7553       551
  human_real     0.8005    0.6044    0.6887       551
    gpt_fake     0.7612    0.9819    0.8576       552
    gpt_real     0.9229    0.6733    0.7786       551

    accuracy                         0.7751      2205
   macro avg     0.7926    0.7750    0.7701      2205
weighted avg     0.7926    0.7751    0.7701      2205


Classification Report:
              precision    recall  f1-score   support

  human_fake     0.8565    0.7477    0.7984       551
  human_real     0.7621    0.8312    0.7951       551
    gpt_fake     0.9232    0.9801    0.9508       552
    gpt_real     0.9255    0.9020    0.9136       551

    accuracy                         0.8653      2205
   macro avg     0.8668    0.8653    0.8645      2205
weighted avg     0.8669    0.8653    0.8645      2205


Classification Report:
              precision    recall  f1-score   support

  h

##CODE SANDBOX

In [ ]:

# # Bal_FNN_df = pd.concat([balanced_df_real,balanced_df_real])

# balanced_df.to_csv('/content/drive/MyDrive/DS266/balgptfake.csv', index=False)



In [ ]:
# augmented_samples = []

# for idx, row in new_FNN_df[new_FNN_df['label'] == 'gpt_real'].iterrows():
#     augmented = augment_gpt_real(row['text'])
#     augmented_samples.append({
#         'text': augmented,
#         'label': row['label'],
#         'original_length': len(row['text'].split()),
#         'new_length': len(augmented.split())
#     })

# # Convert to DataFrame and merge with original
# balanced_df_real = pd.DataFrame(augmented_samples)


# # Verify new length distribution
# print(balanced_df_real.groupby('label')['text'].apply(
#     lambda x: x.str.split().str.len().describe()
# ))

In [ ]:
# balanced_df_real.to_csv('/content/drive/MyDrive/DS266/balanced_df_gptreal.csv', index=False)

In [ ]:

# Bal_FNN_df = pd.concat([balanced_df_real,balanced_df_real])

# Bal_FNN_df.to_csv('/content/drive/MyDrive/DS266/balanced_data.csv', index=False)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')  # First mount your Drive

# # Save as CSV
# balanced_df.to_csv('/content/drive/MyDrive/your_folder/balanced_data.csv', index=False)

In [ ]:
# Bal_FNN_df.to_parquet('/content/drive/MyDrive/DS266/balanced_data_parq.parquet')